### Notebook for training LSTM model with a Kaggke API Key

In [ ]:
from pathlib import Path
import time, json, torch, os

from typing import List, Dict, Tuple
# --- Prologue diagnostic & import path ---
import os, sys, pathlib

print("CWD:", os.getcwd())
print("FILES:", os.listdir())

from main import run_lstm_training
from save_model import resolve_save_dir


In [ ]:
DEFAULT_TRAIN_TICKERS: List[str] = [ "SPY", "ISF.L", "CAC.PA", "EXS1.DE", "IAEX.AS", "1321.T", "XIC.TO", "2800.HK", "STW.AX", "510300.SS",
                                   "IEAG.L","IEAC.AS","EUNH.DE","CBE0.L","IEGS.L","IGLN.L","SSLN.L","CMOD.L","OILB.L",
                                   "ASML.AS","SAP.DE","MC.PA","AIR.PA","OR.PA","SAN.PA","RMS.PA","NESN.SW","ROG.SW","NOVN.SW","SHEL.L","BP.L","TTE.PA",
                                    "QQQ","IWM","VTI","VT","EFA","VEA","EEM","VWO",
                                    "VNQ",                       # REITs
                                    "XLK","XLF","XLV","XLE","XLY","XLP","XLI","XLB","XLC","XLU",  # secteurs
                                    "TLT","IEF","BND","HYG","LQD",  # obligations
                                    "GLD","SLV","DBC","USO",        # matières premières
                                    "AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","BRK-B",
                                    "EWJ","EWG","EWQ","EWA","EWC","EWH","EWT","EWS","EWZ","EZA",
                                    "6758.T","7203.T","9984.T",        # Sony, Toyota, SoftBank
                                    "0700.HK","9988.HK","3690.HK",     # Tencent, Alibaba, Meituan
                                    "RY.TO","TD.TO","SHOP.TO","ENB.TO","BHP.AX","CBA.AX"# Canada & Australie
                                   ]

#DEFAULT_TRAIN_TICKERS: List[str] = [ "SPY", "ISF.L", "CAC.PA", "EXS1.DE", "IAEX.AS", "1321.T", "XIC.TO", "2800.HK", "STW.AX", "510300.SS"]
FOREX_TICKERS: Dict[str, str] = {
    "EURUSD=X": "FX_EURUSD",
    "GBPUSD=X": "FX_GBPUSD",
    "USDJPY=X": "FX_USDJPY",
}

LSTM_FEATURE_COLUMNS: List[str] = [
    "volume_log",
    "ret",
    "ret_mean_5",
    "ret_mean_20",
    "ret_mean_50",
    "ret_std_20",
    "ret_std_50",
    "price_zscore_20",
    "RSI_14",
] + [f"{alias}_RET" for alias in FOREX_TICKERS.values()]

TARGET_COLUMN: str = "ret"

hp = {
    "window_size": 60, #100
    "hidden_size": 48, #64
    "num_layers": 1, #2
    "lr": 1e-3,
    "epochs": 700, #200
    "horizon": 10,
    "residual_boosting": False,
}

In [ ]:
# === À faire AVANT l'entraînement ===
save_dir = resolve_save_dir("Forecast", file='models_saved')
best_path = save_dir / "best.pt"   # meilleur modèle (val la + basse)
last_path = save_dir / "last.pt"   # dernier état (pour reprise)
print("Saving to:", save_dir)

In [ ]:
run_lstm_training(
    hp=hp,
    save_dir=save_dir,
    tickers= DEFAULT_TRAIN_TICKERS,
    period = "max",
    interval = "1d",
    plot_training = False,
    plot_dir  = False
)